# EMA Touch — EMA Support / Resistance Selection

## Contents

- [Configuration](#configuration)
  - [Setup](#setup)
  - [Automatic](#automatic)
  - [Manual](#manual)
  - [Final configuration](#final-configuration)
- [Exploratory EMA statistics](#exploratory-ema-statistics)
- [Support and resistance analysis](#support-and-resistance-analysis)
  - [Filters](#filters)
  - [EMA as support](#ema-as-support)
  - [EMA as resistance](#ema-as-resistance)
  - [Universal EMA](#universal-ema)
- [Visual analysis](#visual-analysis)
- [Conclusion](#conclusion)

Ranks every EMA period by how price interacts with it — as support, as resistance, and as either (universal) — so you can choose the entry EMA for the ema_touch strategy on evidence rather than habit.

The ema_touch strategy enters on a wick that tags an EMA within a delta tolerance and closes back on the trend side. Which EMA to tag is exactly the question this notebook answers: it counts, per EMA, how often price touched, crossed, held, or broke it, then ranks the periods by five support metrics, five resistance metrics, and three universal metrics.

Two filters (cross-frequency and sample-size) suppress structurally degenerate or under-sampled EMAs. The Conclusion turns the top-ranked periods straight into an EmaTouchParams you can run.

Data, the touch tolerance, and the search range all come from the project configurators, so this analysis stays in step with what the strategy actually trades.

How the per-EMA analysis works:
- Every evaluated candle is placed in exactly one mutually exclusive bucket relative to the EMA:
    - cross: low <= EMA <= high (the bar's range straddles the EMA)
    - low_touch: the whole bar is above the EMA and its low is within delta of it (a touch from above)
    - high_touch: the whole bar is below the EMA and its high is within delta of it (a touch from below)
    - above: the whole bar is above the EMA and farther than delta (clean uptrend bar)
    - below: the whole bar is below the EMA and farther than delta (clean downtrend bar)
- A cross's direction is resolved by the bar's open relative to the EMA: open above the EMA is a support test (price came down into it), open below is a resistance test (price came up into it).
- A test held when the close finished back on the supported side (support: close > EMA; resistance: close < EMA). A close exactly at the EMA counts as neither held nor broken, which avoids the EMA-1 degeneracy where close == EMA on every bar.
- delta units follow delta_mode: absolute (quote points) or percent (% of the EMA). This mirrors ema_touch's ema_touch_delta / ema_touch_delta_mode.

Invariants that always hold: cross + low_touch + high_touch + above + below = evaluated_candles; any_touch = low_touch + high_touch; support_held <= support_test; resistance_held <= resistance_test.

## Configuration

### Setup

In [ ]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [ ]:
import dataclasses

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from engine.data_configurator import ACTIVE, load_data
from engine.strategy_configurator import params_for
from engine.indicators import ema

In [ ]:
# Analysis helpers (research-only, so they live in the notebook rather than the
# tested engine surface). The EMA itself comes from engine.indicators.ema, so the
# ranking is computed on the exact moving average the engine and ema_touch use.

def analyze_ema_touches(df, ema_range, delta, delta_mode, *, skip_warmup=True):
    """Count touch / cross / above / below behaviour of every EMA in ema_range.

    Returns one row per EMA period with the touch buckets plus the support /
    resistance test-and-held counters used by the ratios below. skip_warmup
    drops the first `period` bars of each EMA while it is still warming up.
    """
    close, high, low, open_ = df["close"], df["high"], df["low"], df["open"]
    rows = []
    for period in ema_range:
        e = ema(close, period)
        if delta_mode == "percent":
            tol = e.abs() * (delta / 100.0)
        elif delta_mode == "absolute":
            tol = pd.Series(delta, index=e.index)
        else:
            raise ValueError("delta_mode must be 'percent' or 'absolute'")

        warmup = period if skip_warmup else 0
        mask = pd.Series(False, index=e.index)
        mask.iloc[warmup:] = True

        crossed = (low <= e) & (e <= high) & mask
        strictly_above = (low > e) & mask
        strictly_below = (high < e) & mask

        # Touches are near approaches that did not cross.
        low_touch = strictly_above & ((low - e) <= tol)
        high_touch = strictly_below & ((e - high) <= tol)
        any_touch = low_touch | high_touch

        # Clean trending bars (past the EMA by more than delta).
        above = strictly_above & ~low_touch
        below = strictly_below & ~high_touch

        # Cross direction by the open; held by the close. Strict inequalities,
        # so a tie at the EMA counts as neither held nor broken.
        crossed_from_above = crossed & (open_ > e)
        crossed_from_below = crossed & (open_ < e)
        crossed_held_above = crossed_from_above & (close > e)
        crossed_held_below = crossed_from_below & (close < e)

        support_test = low_touch | crossed_from_above
        support_held = low_touch | crossed_held_above
        resistance_test = high_touch | crossed_from_below
        resistance_held = high_touch | crossed_held_below

        rows.append({
            "ema": period,
            "low_touch": int(low_touch.sum()),
            "high_touch": int(high_touch.sum()),
            "any_touch": int(any_touch.sum()),
            "above": int(above.sum()),
            "below": int(below.sum()),
            "cross": int(crossed.sum()),
            "cross_above": int(crossed_from_above.sum()),
            "cross_below": int(crossed_from_below.sum()),
            "support_test": int(support_test.sum()),
            "support_held": int(support_held.sum()),
            "resistance_test": int(resistance_test.sum()),
            "resistance_held": int(resistance_held.sum()),
            "evaluated_candles": int(mask.sum()),
        })
    return pd.DataFrame(rows)


def filter_by_cross_rate(df, max_cross_rate):
    """Add a cross_rate column and keep only EMAs below the threshold.

    Drops EMAs that hug price too tightly to act as support/resistance (a fast
    EMA's range crosses on most bars, making the close-direction checks trivial).
    """
    df = df.copy()
    df["cross_rate"] = df["cross"] / df["evaluated_candles"]
    return df[df["cross_rate"] < max_cross_rate]

### Automatic

In [ ]:
# Automatic config: the dataset comes from the project-wide data configurator;
# the touch tolerance is seeded from the ema_touch strategy defaults so the
# ranking is measured at the same delta the strategy trades.
DATA_CONFIG = ACTIVE                       # engine/data_configurator.py (DataSpec)

_ema_touch = params_for("ema_touch")       # engine/strategy_configurator.py (EmaTouchParams)
DELTA      = _ema_touch.ema_touch_delta        # touch tolerance magnitude
DELTA_MODE = _ema_touch.ema_touch_delta_mode   # "absolute" (quote points) | "percent" (% of EMA)

# Analysis search + filter knobs.
EMA_RANGE       = range(1, 200)            # EMA periods to evaluate
SKIP_WARMUP     = True                     # drop each EMA's first `period` warm-up bars
MAX_CROSS_RATE  = 0.3                      # cross-frequency filter: keep EMAs crossed < 30% of bars
MIN_TOUCHES_SUP = 30                       # sample-size filter (support tests)
MIN_TOUCHES_RES = 30                       # sample-size filter (resistance tests)
MIN_TOUCHES_UNI = 60                       # sample-size filter (support + resistance tests)

# The ranking supports delta_mode "absolute" or "percent". If the strategy default
# is "atr" (cross-symbol mode), fall back to absolute points here so the cell runs.
if DELTA_MODE not in ("absolute", "percent"):
    DELTA, DELTA_MODE = 40.0, "absolute"

### Manual

Automatic defaults, with any Manual overrides layered on top:
- Leave the override lines commented out to keep the Automatic defaults.
- Uncomment a line to override that one knob.

In [ ]:
# manual override — DATA. Fill the dict to override; empty = automatic (ACTIVE).
DATA_OVERRIDES = {}          # e.g. {"symbol": "ETHUSDT", "interval": "60", "num_candles": 1500}
DATA_CONFIG = dataclasses.replace(DATA_CONFIG, **DATA_OVERRIDES)

In [ ]:
# manual override — ANALYSIS knobs. Uncomment a line to override the Automatic default.
# EMA_RANGE       = range(1, 300)
# DELTA           = 30.0
# DELTA_MODE      = "percent"     # "absolute" (quote points) | "percent" (% of EMA)
# MAX_CROSS_RATE  = 0.2           # stricter: keep only slower, decisive S/R levels
# MIN_TOUCHES_SUP = 40
# MIN_TOUCHES_RES = 40
# MIN_TOUCHES_UNI = 80
pass

### Final configuration

In [ ]:
# Load candles from the project cache and run the per-EMA support/resistance
# analysis. This is the analysis analogue of the strategy notebooks' final-config
# cell: everything below uses df, SYMBOL, INTERVAL, result, and the filter knobs.
df = load_data(DATA_CONFIG)
SYMBOL, INTERVAL = DATA_CONFIG.symbol, DATA_CONFIG.interval
result = analyze_ema_touches(df, EMA_RANGE, DELTA, DELTA_MODE, skip_warmup=SKIP_WARMUP)

_window = (f"{DATA_CONFIG.start} -> {DATA_CONFIG.end or 'now'}"
           if DATA_CONFIG.is_range else f"last {DATA_CONFIG.num_candles}")
print(f"Loaded {len(df):,} candles | {SYMBOL} {INTERVAL}m {DATA_CONFIG.category} | "
      f"{_window} | {df.index[0]:%Y-%m-%d %H:%M} -> {df.index[-1]:%Y-%m-%d %H:%M} UTC")
print(f"Analysed {len(result)} EMAs | delta={DELTA} ({DELTA_MODE}) | "
      f"range={EMA_RANGE.start}..{EMA_RANGE.stop - 1}")
print(f"Filters: MAX_CROSS_RATE={MAX_CROSS_RATE} | "
      f"MIN_TOUCHES support={MIN_TOUCHES_SUP}, resistance={MIN_TOUCHES_RES}, universal={MIN_TOUCHES_UNI}")

## Exploratory EMA statistics

__Distributions of EMA periods across candle categories.__

EMA periods are grouped by their count for each candle category.

For each category, the histogram shows how the per-EMA count is distributed across the EMA range — how many EMAs had roughly 100 touches, 500, 2000, and so on.

Use it to see the bulk versus the tails: where most EMAs cluster (typical behaviour) and which ones sit far from the pack (outliers worth a closer look).

Chart explanation:
- each panel counts EMA periods, not candles
- the x-axis is the number of times this candle behaviour happened
- the y-axis is how many EMAs had that number
- a tall bar in the middle means most EMAs behave this way (the typical range)
- short bars far on the sides are a few EMAs whose counts are unusually high or low (the outliers)

For example, the low_touch panel answers how many EMAs in the range had around 100 low touches, around 300, around 400 — each EMA has one low_touch count, and the histogram bins those counts.

In [ ]:
# Per-EMA count distributions, one panel per candle category (Plotly faceted).
cats = ["low_touch", "high_touch", "any_touch", "cross", "above", "below"]
fig = make_subplots(rows=2, cols=3, subplot_titles=cats)
for k, col in enumerate(cats):
    r, c = divmod(k, 3)
    fig.add_trace(
        go.Histogram(x=result[col], nbinsx=50, marker_color="steelblue",
                     name=col, showlegend=False),
        row=r + 1, col=c + 1,
    )
fig.update_layout(
    title="Distribution of per-EMA counts by candle category",
    template="plotly_white", height=600, bargap=0.05,
)
fig.update_xaxes(title_text="count for this EMA")
fig.update_yaxes(title_text="number of EMAs")
fig.show()

__Which EMA periods are touched most often by candle lows?__

In [ ]:
emas_sorted_by_lows = (result[["ema", "low_touch"]]
           .sort_values("low_touch", ascending=False)
           .set_index("ema"))

emas_sorted_by_lows

__Which EMA periods are touched most often by candle highs?__

In [ ]:
emas_sorted_by_highs = (result[["ema", "high_touch"]]
           .sort_values("high_touch", ascending=False)
           .set_index("ema"))

emas_sorted_by_highs

__Which EMA periods have the most candles floating entirely above them?__

Price spent most of the period trending above this EMA (the EMA acted as a floor / bullish regime).

In [ ]:
emas_sorted_by_above = (result[["ema", "above"]]
           .sort_values("above", ascending=False)
           .set_index("ema"))

emas_sorted_by_above

__Which EMA periods have the most candles floating entirely below them?__

Price spent most of the period trending below this EMA (the EMA acted as a ceiling / bearish regime).

In [ ]:
emas_sorted_by_below = (result[["ema", "below"]]
           .sort_values("below", ascending=False)
           .set_index("ema"))

emas_sorted_by_below

## Support and resistance analysis

### Filters

Two filters for opposite EMA extremes:
- cross-frequency: drop EMAs that price crosses too often
- sample-size: drop EMAs that price tests too rarely

What is left in the middle is the tradeable zone — EMAs that price interacts with often enough to evaluate, but not so often that the EMA just tracks price.

__*1. Cross-frequency filter*__

Filters out EMAs that are crossed too frequently.

"Is this EMA structurally able to act as support/resistance at all?"

If price crosses an EMA on most bars, the EMA is not a wall — it is tracking price. Very fast EMAs stick too closely to price to act as meaningful support or resistance, so conditions like close >= EMA or close <= EMA become trivial and misleading.

At the extreme, an EMA with period 1 (ema span 1 equals the close) is identical to price. With strict-inequality held checks, every bar's close equals the EMA, so neither held nor broken fires and the support ratio collapses to zero regardless of price action. The cross-saturation guard removes such EMAs before they can mislead.

Any EMA whose cross rate (cross / evaluated_candles) is at or above MAX_CROSS_RATE is excluded from the ratio calculations:
- 0.5 keeps EMAs crossed in less than 50% of candles
- 0.3 keeps EMAs crossed in less than 30% of candles
- 0.2 keeps only slower EMAs that behave as strong, decisive S/R levels

Lower threshold means a stricter filter and fewer, more meaningful EMAs.

__*2. Sample-size filter*__

Do we have enough data to trust the ratio?

A 100% hold rate over 3 tests is noise. A 70% hold rate over 200 tests is a real signal. The filter requires a minimum number of tests before an EMA's ratio is considered meaningful.

Rule of thumb: at least about one touch per week of data. For example, 8640 candles of 15-minute data is about 90 days, so MIN_TOUCHES = 30 is about one touch every three days — a reasonable minimum.

The threshold can be fixed (a number that feels statistically meaningful) or data-driven (auto-scale to the median or a quantile of the test counts — see the commented options in the cell below). It is set separately for support, resistance, and universal EMAs.

__Filter thresholds__

In [ ]:
# The filter thresholds are defined once in the Configuration chapter (Automatic /
# Manual) and reused by every ratio below. Echo the active values here.
print(f"MAX_CROSS_RATE  = {MAX_CROSS_RATE}")
print(f"MIN_TOUCHES_SUP = {MIN_TOUCHES_SUP}")
print(f"MIN_TOUCHES_RES = {MIN_TOUCHES_RES}")
print(f"MIN_TOUCHES_UNI = {MIN_TOUCHES_UNI}")

# Data-driven alternatives (uncomment in the Manual cell to use instead of fixed values):
#   MIN_TOUCHES_SUP = int(result["support_test"].median())
#   MIN_TOUCHES_RES = int(result["resistance_test"].median())
#   MIN_TOUCHES_UNI = int((result["support_test"] + result["resistance_test"]).median())

### EMA as support

High ratio means the EMA acted as support: price dipped into it, touched or briefly crossed, but the close finished back above without committing to a breakdown.

Adjacent EMAs clustering with nearly identical ratios suggest a support zone was present in the analysed sample. This describes past data and does not imply the zone will continue to hold.

Which formula to use?

| # | formula | meaning | what it measures | usage |
|---|---|---|---|---|
| 1 | support_held / support_test | hold rate: of all support tests, share that held | When this EMA is tested from above (touched or crossed), how often does it hold (close back above)? | support-quality metric |
| 2 | low_touch / support_test | rejection rate: of all support tests, share that rejected cleanly | How clean is this EMA's support (pure wick rejections; any pierce is a failure even if it recovered)? | stricter support-quality metric |
| 3 | low_touch / evaluated_candles | frequency of clean support tests | How often does price reach this EMA and produce a tradeable support setup? | tradability, activity gauge |
| 4 | (low_touch + above) / cross | bullishness at this EMA | For every pierce, how many bars stayed entirely above? | regime / trend-bias ratio |
| 5 | (low_touch + above) / evaluated_candles | regime indicator | Fraction of bars where the entire candle was above the EMA | trend filter |

__*1. Hold rate: is this EMA a good support level to trade?*__

ratio_support_1 is the share of support tests that held:
- 0.0 means every approach broke through (no holds)
- 0.5 means bounces half the time when tested
- 1.0 means every approach held (perfect support)

A support test is a bar that approached the EMA from above (a low_touch, or a cross that opened above the EMA). It held when the bar did not end below the EMA (close > EMA). A close exactly at the EMA counts as neither held nor broken.

In [ ]:
# The unfiltered table. replace(0, np.nan) guards against an EMA with no support
# tests at all; .copy() keeps edits local.
ratio_support_1 = result[["ema", "support_test", "support_held", "cross", "evaluated_candles"]].copy()
ratio_support_1["ratio_support_1"] = ratio_support_1["support_held"] / ratio_support_1["support_test"].replace(0, np.nan)
ratio_support_1 = ratio_support_1.sort_values("ratio_support_1", ascending=False).set_index("ema")

ratio_support_1

Notes:
- Slow EMAs (above about 100) often score highest because price rarely reaches them; when they are tested the test tends to be a wick rejection rather than a clean breakdown.
- Always weight the ranking by sample size (support_test).
- Re-run on different windows to see how stable each EMA's hold rate is.

__Filtering out the results__

In [ ]:
# Top 10 EMAs that pass both filters: cross-frequency (structural) and sample-size (statistical).
ratio_support_1_filtered = filter_by_cross_rate(ratio_support_1, MAX_CROSS_RATE)
ratio_support_1_filtered = ratio_support_1_filtered[ratio_support_1_filtered["support_test"] >= MIN_TOUCHES_SUP]
ratio_support_1_filtered.head(10)

__*2. Rejection rate: how clean is this EMA's support?*__

Stricter metric: of all approaches from above, what fraction were clean wick rejections that did not pierce? The denominator support_test = low_touch + cross_from_above by construction.

In [ ]:
ratio_support_2 = result[["ema", "low_touch", "support_test", "cross", "evaluated_candles"]].copy()
ratio_support_2["ratio_support_2"] = ratio_support_2["low_touch"] / ratio_support_2["support_test"].replace(0, np.nan)
ratio_support_2 = ratio_support_2.sort_values("ratio_support_2", ascending=False).set_index("ema")

ratio_support_2

__Filtering out the results__

In [ ]:
ratio_support_2_filtered = filter_by_cross_rate(ratio_support_2, MAX_CROSS_RATE)
ratio_support_2_filtered = ratio_support_2_filtered[ratio_support_2_filtered["support_test"] >= MIN_TOUCHES_SUP]
ratio_support_2_filtered

__*3. Tradability: how often does price reach this EMA?*__

Evaluates how often the setup shows up, not how good it is when it does.

In [ ]:
ratio_support_3 = result[["ema", "low_touch", "cross", "evaluated_candles"]].copy()
ratio_support_3["ratio_support_3"] = ratio_support_3["low_touch"] / ratio_support_3["evaluated_candles"].replace(0, np.nan)
ratio_support_3 = ratio_support_3.sort_values("ratio_support_3", ascending=False).set_index("ema")

ratio_support_3

__*4. Bullishness: what is the trend bias around this EMA?*__

For every time price pierced the EMA (in either direction), how many bars sat entirely above it? Captures EMAs that mark decisive, uninterrupted bullish regimes — a floor that was rarely breached.

Example: (low_touch + above) / cross = 850 / 150 = 5.67 means about 5.7 bars sat entirely above for every pierce — a strong bullish regime.

In [ ]:
ratio_support_4 = result[["ema", "low_touch", "above", "cross"]].copy()
ratio_support_4["ratio_support_4"] = (ratio_support_4["low_touch"] + ratio_support_4["above"]) / ratio_support_4["cross"].replace(0, np.nan)
ratio_support_4 = ratio_support_4.sort_values("ratio_support_4", ascending=False).set_index("ema")

ratio_support_4

__*5. Trend filter: fraction of bars where the entire candle was above the EMA.*__

A regime indicator: a high ratio means price spent the period mostly above the EMA. Rewards EMAs that acted as a floor during trending stretches where price never dipped close enough to register a touch. Useful in strong uptrends where pure touch-based ratios under-count respected levels.

In [ ]:
ratio_support_5 = result[["ema", "low_touch", "above", "cross", "evaluated_candles"]].copy()
ratio_support_5["ratio_support_5"] = (ratio_support_5["low_touch"] + ratio_support_5["above"]) / ratio_support_5["evaluated_candles"].replace(0, np.nan)
ratio_support_5 = ratio_support_5.sort_values("ratio_support_5", ascending=False).set_index("ema")

ratio_support_5

__Visualization: hold rate (support quality) by EMA period.__

A dual-axis chart: stacked bars for held versus broken counts, and a red line for the hold rate. Total bar height is the total support tests for that EMA. broken = support_test - support_held; hold rate = support_held / support_test.

Read it for where the red line peaks across the EMA spectrum (the best support periods) and whether the peaks are isolated spikes or broad hills (a whole zone respecting support).

In [ ]:
# Plot only EMAs that pass both the cross-frequency and sample-size filters.
viz = ratio_support_1_filtered.reset_index().sort_values("ema")

ratio = viz["ratio_support_1"]
broke = viz["support_test"] - viz["support_held"]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(x=viz["ema"], y=viz["support_held"], name="support held",
           marker_color="orange", opacity=0.75),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(x=viz["ema"], y=broke, name="support broke",
           marker_color="steelblue", opacity=0.45),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=viz["ema"], y=ratio, name="support rate",
               mode="lines", line=dict(color="red", width=2),
               hovertemplate="%{y:.1%}"),
    secondary_y=True,
)
fig.update_layout(
    title="Support held vs broken by EMA period — with hold rate",
    barmode="stack", template="plotly_white", height=500,
    hovermode="x unified", hoverlabel=dict(namelength=-1),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_xaxes(title_text="EMA period", tickprefix="EMA ")
fig.update_yaxes(title_text="Count", secondary_y=False)
fig.update_yaxes(title_text="Support rate", secondary_y=True, color="red", tickformat=".0%")
fig.show()

### EMA as resistance

High ratio means the EMA acted as resistance: price rallied into it, the high kissed or briefly crossed, and the close printed back below without committing to a breakout.

Adjacent EMAs clustering with nearly identical ratios suggest a resistance zone was present in the analysed sample. This describes past data and does not imply the zone will continue to hold.

Common expectations:
- On a 15-minute chart, slow EMAs like 99, 144 and 200 often score high because price rarely reaches them and rejects cleanly when it does.
- In a downtrending or ranging market, the classic respected EMAs (20, 50, 200) tend to rise to the top.
- In a strong uptrend, resistance EMAs barely exist: price stays above most EMAs, high_touch is low across the board, and the ratio becomes noisy, often dominated by slow EMAs with tiny sample sizes.

Which formula to use?

| # | formula | meaning | what it measures | usage |
|---|---|---|---|---|
| 1 | resistance_held / resistance_test | hold rate: of all resistance tests, share that held | When this EMA is tested from below (touched or crossed), how often does it hold (close back below)? | resistance-quality metric |
| 2 | high_touch / resistance_test | rejection rate: of all resistance tests, share that rejected cleanly | How clean is this EMA's resistance (pure wick rejections; any pierce is a failure even if it recovered)? | stricter resistance-quality metric |
| 3 | high_touch / evaluated_candles | frequency of clean resistance tests | How often does price reach this EMA and produce a tradeable resistance setup? | tradability, activity gauge |
| 4 | (high_touch + below) / cross | bearishness at this EMA | For every pierce, how many bars stayed entirely below? | regime / trend-bias ratio |
| 5 | (high_touch + below) / evaluated_candles | regime indicator | Fraction of bars where the entire candle was below the EMA | trend filter |

__*1. Hold rate: is this EMA a good resistance level to trade?*__

ratio_resistance_1 is the share of resistance tests that held:
- 0.0 means every approach broke through (no holds)
- 0.5 means rejects half the time when tested
- 1.0 means every approach held (perfect resistance)

A resistance test is a bar that approached the EMA from below (a high_touch, or a cross that opened below the EMA). It held when the bar did not end above the EMA (close < EMA). A close exactly at the EMA counts as neither held nor broken.

In [ ]:
# The unfiltered table.
ratio_resistance_1 = result[["ema", "resistance_test", "resistance_held", "cross", "evaluated_candles"]].copy()
ratio_resistance_1["ratio_resistance_1"] = ratio_resistance_1["resistance_held"] / ratio_resistance_1["resistance_test"].replace(0, np.nan)
ratio_resistance_1 = ratio_resistance_1.sort_values("ratio_resistance_1", ascending=False).set_index("ema")

ratio_resistance_1

__Filtering out the results__

In [ ]:
# Top 10 EMAs that pass both filters: cross-frequency (structural) and sample-size (statistical).
ratio_resistance_1_filtered = filter_by_cross_rate(ratio_resistance_1, MAX_CROSS_RATE)
ratio_resistance_1_filtered = ratio_resistance_1_filtered[ratio_resistance_1_filtered["resistance_test"] >= MIN_TOUCHES_RES]
ratio_resistance_1_filtered.head(10)

__*2. Rejection rate: how clean is this EMA's resistance?*__

Stricter metric: of all approaches from below, what fraction were clean wick rejections that did not pierce? The denominator resistance_test = high_touch + cross_from_below by construction.

In [ ]:
ratio_resistance_2 = result[["ema", "high_touch", "resistance_test", "cross", "evaluated_candles"]].copy()
ratio_resistance_2["ratio_resistance_2"] = ratio_resistance_2["high_touch"] / ratio_resistance_2["resistance_test"].replace(0, np.nan)
ratio_resistance_2 = ratio_resistance_2.sort_values("ratio_resistance_2", ascending=False).set_index("ema")

ratio_resistance_2

__Filtering out the results__

In [ ]:
ratio_resistance_2_filtered = filter_by_cross_rate(ratio_resistance_2, MAX_CROSS_RATE)
ratio_resistance_2_filtered = ratio_resistance_2_filtered[ratio_resistance_2_filtered["resistance_test"] >= MIN_TOUCHES_RES]
ratio_resistance_2_filtered

__*3. Tradability: how often does price reach this EMA?*__

Evaluates how often the setup shows up, not how good it is when it does.

In [ ]:
ratio_resistance_3 = result[["ema", "high_touch", "cross", "evaluated_candles"]].copy()
ratio_resistance_3["ratio_resistance_3"] = ratio_resistance_3["high_touch"] / ratio_resistance_3["evaluated_candles"].replace(0, np.nan)
ratio_resistance_3 = ratio_resistance_3.sort_values("ratio_resistance_3", ascending=False).set_index("ema")

ratio_resistance_3

__*4. Bearishness: what is the trend bias around this EMA?*__

For every time price pierced the EMA (in either direction), how many bars sat entirely below it? Captures EMAs that mark decisive, uninterrupted bearish regimes — a ceiling that was rarely breached.

Example: (high_touch + below) / cross = 850 / 150 = 5.67 means about 5.7 bars sat entirely below for every pierce — a strong bearish regime.

In [ ]:
ratio_resistance_4 = result[["ema", "high_touch", "below", "cross"]].copy()
ratio_resistance_4["ratio_resistance_4"] = (ratio_resistance_4["high_touch"] + ratio_resistance_4["below"]) / ratio_resistance_4["cross"].replace(0, np.nan)
ratio_resistance_4 = ratio_resistance_4.sort_values("ratio_resistance_4", ascending=False).set_index("ema")

ratio_resistance_4

__*5. Trend filter: fraction of bars where the entire candle was below the EMA.*__

A regime indicator: a high ratio means price spent the period mostly below the EMA. Rewards EMAs that acted as a ceiling during trending stretches where price never rallied close enough to register a touch. Useful in strong downtrends where pure touch-based ratios under-count respected levels.

In [ ]:
ratio_resistance_5 = result[["ema", "high_touch", "below", "cross", "evaluated_candles"]].copy()
ratio_resistance_5["ratio_resistance_5"] = (ratio_resistance_5["high_touch"] + ratio_resistance_5["below"]) / ratio_resistance_5["evaluated_candles"].replace(0, np.nan)
ratio_resistance_5 = ratio_resistance_5.sort_values("ratio_resistance_5", ascending=False).set_index("ema")

ratio_resistance_5

__Visualization: hold rate (resistance quality) by EMA period.__

A dual-axis chart: stacked bars for held versus broken counts, and a red line for the hold rate. Total bar height is the total resistance tests for that EMA. broken = resistance_test - resistance_held; hold rate = resistance_held / resistance_test.

Read it for where the red line peaks across the EMA spectrum (the best resistance periods) and whether the peaks are isolated spikes or broad hills.

In [ ]:
# Plot only EMAs that pass both the cross-frequency and sample-size filters.
viz = ratio_resistance_1_filtered.reset_index().sort_values("ema")

ratio = viz["ratio_resistance_1"]
broke = viz["resistance_test"] - viz["resistance_held"]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(x=viz["ema"], y=viz["resistance_held"], name="resistance held",
           marker_color="orange", opacity=0.75),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(x=viz["ema"], y=broke, name="resistance broke",
           marker_color="steelblue", opacity=0.45),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=viz["ema"], y=ratio, name="resistance rate",
               mode="lines", line=dict(color="red", width=2),
               hovertemplate="%{y:.1%}"),
    secondary_y=True,
)
fig.update_layout(
    title="Resistance held vs broken by EMA period — with hold rate",
    barmode="stack", template="plotly_white", height=500,
    hovermode="x unified", hoverlabel=dict(namelength=-1),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_xaxes(title_text="EMA period", tickprefix="EMA ")
fig.update_yaxes(title_text="Count", secondary_y=False)
fig.update_yaxes(title_text="Resistance rate", secondary_y=True, color="red", tickformat=".0%")
fig.show()

### Universal EMA

EMA as any kind of support or resistance. Three metrics are reported:
- Hold rate: quality across both directions.
- Bounce rate: stricter restraint — the fraction of interactions that did not pierce.
- Tradability: frequency of any S/R setup.

Flow: best metric, then strict alternative, then frequency check.

Which formula to use?

| # | formula | meaning | what it measures | usage |
|---|---|---|---|---|
| 1 | (support_held + resistance_held) / (support_test + resistance_test) | hold rate: of all S/R tests, what fraction held | When this EMA is tested from either side, how often does the close finish back on the supported side? | universal-quality metric |
| 2 | any_touch / (any_touch + cross) | bounce rate: fraction of interactions that did not pierce | Does this EMA act as a wall, regardless of direction? | stricter bounding metric |
| 3 | any_touch / evaluated_candles | frequency of clean S/R touches | How often does price reach this EMA at all? | tradability, activity gauge |

__*1. Hold rate: is this EMA a good universal S/R level to trade?*__

Weighted average of the support and resistance hold rates. ratio_universal_1 is the share of S/R tests (in either direction) that held:
- 0.0 means every test broke through
- 0.5 means bounces half the time when tested
- 1.0 means every test held

The question: when price tests this EMA, how often does it get pushed back?

In [ ]:
# Cross direction is resolved by the open (open > EMA is a support test, open < EMA
# a resistance test), so each cross is counted in exactly one direction. Within that
# direction, held vs broken is decided by the close. Pure touches always hold.
ratio_universal_1 = result[["ema", "support_test", "support_held",
                            "resistance_test", "resistance_held",
                            "cross", "evaluated_candles"]].copy()
ratio_universal_1["ratio_universal_1"] = (
    (ratio_universal_1["support_held"] + ratio_universal_1["resistance_held"])
    / (ratio_universal_1["support_test"] + ratio_universal_1["resistance_test"]).replace(0, np.nan)
)
ratio_universal_1 = ratio_universal_1.sort_values("ratio_universal_1", ascending=False).set_index("ema")

ratio_universal_1

__Filtering out the results__

In [ ]:
# Top 10 EMAs that pass both filters. Sample size = total S/R tests across both directions.
ratio_universal_1_filtered = filter_by_cross_rate(ratio_universal_1, MAX_CROSS_RATE)
total_test = ratio_universal_1_filtered["support_test"] + ratio_universal_1_filtered["resistance_test"]
ratio_universal_1_filtered = ratio_universal_1_filtered[total_test >= MIN_TOUCHES_UNI]
ratio_universal_1_filtered.head(10)

__*2. Bounce rate: does this EMA act as a wall?*__

Strict restraint metric: of all interactions with the EMA (touches or crosses), what fraction did not pierce through? ratio_universal_2:
- 0.0 means every interaction was a pierce
- 0.5 means half the interactions were wick-only
- 1.0 means no pierces (perfect wall)

It does not depend on close direction, only on whether the bar's range straddled the EMA. EMAs that get touched but not pierced score high regardless of trend.

In [ ]:
ratio_universal_2 = result[["ema", "any_touch", "cross", "evaluated_candles"]].copy()
ratio_universal_2["ratio_universal_2"] = ratio_universal_2["any_touch"] / (ratio_universal_2["any_touch"] + ratio_universal_2["cross"]).replace(0, np.nan)
ratio_universal_2 = ratio_universal_2.sort_values("ratio_universal_2", ascending=False).set_index("ema")

ratio_universal_2

__Filtering out the results__

In [ ]:
# Sample size = bounce rate's denominator (any_touch + cross), not any_touch alone.
ratio_universal_2_filtered = filter_by_cross_rate(ratio_universal_2, MAX_CROSS_RATE)
ratio_universal_2_filtered = ratio_universal_2_filtered[
    (ratio_universal_2_filtered["any_touch"] + ratio_universal_2_filtered["cross"]) >= MIN_TOUCHES_UNI
]
ratio_universal_2_filtered

__*3. Tradability: how often does price reach this EMA?*__

Evaluates how often the EMA produces any S/R setup (a touch from above or below), not how good those setups are. ratio_universal_3 is the fraction of bars that registered as a clean S/R touch (no piercing):
- high ratio means a frequently tested EMA with lots of opportunities
- low ratio means a rarely tested EMA (slow EMAs often live here)

Useful as an activity gauge: an EMA with high quality but very low tradability rarely fires.

In [ ]:
ratio_universal_3 = result[["ema", "any_touch", "cross", "evaluated_candles"]].copy()
ratio_universal_3["ratio_universal_3"] = ratio_universal_3["any_touch"] / ratio_universal_3["evaluated_candles"].replace(0, np.nan)
ratio_universal_3 = ratio_universal_3.sort_values("ratio_universal_3", ascending=False).set_index("ema")

ratio_universal_3

__Visualization: hold rate (universal quality) by EMA period.__

A dual-axis chart across both directions: stacked bars for held versus broken counts, and a red line for the universal hold rate. Total bar height is the total S/R tests for that EMA. broken = (support_test + resistance_test) - (support_held + resistance_held); hold rate = (support_held + resistance_held) / (support_test + resistance_test).

What to read:
- Stacked bars are the total S/R tests per EMA, split into held (orange) and broken (blue). Tall bars are heavily tested EMAs; short bars are rarely tested (treat their ratios with caution).
- The red line is the universal hold rate.
- Fast EMAs (left, after the cross-saturation filter) carry meaningful test counts but rarely hold cleanly, so the line sits lower.
- Medium EMAs (middle) are often the sweet spot: enough tests to be meaningful, with the hold rate peaking here.
- Slow EMAs (right) shrink in absolute count and the hold rate becomes noisy.

In [ ]:
# Plot only EMAs that pass both the cross-frequency and sample-size filters.
viz = ratio_universal_1_filtered.reset_index().sort_values("ema")

held = viz["support_held"] + viz["resistance_held"]
total = viz["support_test"] + viz["resistance_test"]
broke = total - held
ratio = viz["ratio_universal_1"]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Bar(x=viz["ema"], y=held, name="held (support + resistance)",
           marker_color="orange", opacity=0.75),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(x=viz["ema"], y=broke, name="broken",
           marker_color="steelblue", opacity=0.45),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=viz["ema"], y=ratio, name="universal hold rate",
               mode="lines", line=dict(color="red", width=2),
               hovertemplate="%{y:.1%}"),
    secondary_y=True,
)
fig.update_layout(
    title="Universal quality by EMA period — held vs broken with hold rate",
    barmode="stack", template="plotly_white", height=500,
    hovermode="x unified", hoverlabel=dict(namelength=-1),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig.update_xaxes(title_text="EMA period", tickprefix="EMA ")
fig.update_yaxes(title_text="Count", secondary_y=False)
fig.update_yaxes(title_text="Universal hold rate", secondary_y=True, color="red", tickformat=".0%")
fig.show()

## Visual analysis

An interactive candlestick chart that overlays chosen EMAs, to eyeball whether they line up with swing highs and lows in the analysed window.
- The chart opens on the last 3 days. Drag the range slider or use the 1d / 1w / 1m / 3m buttons to move.
- After jumping, click Autoscale in the top-right toolbar (or double-click the y-axis) to rescale.
- Change initial_window to days=1 for a tighter zoom or days=7 for a weekly view.

__Configurable parameters__ are in the next cell.

In [ ]:
ema_periods    = [53, 77, 84, 114]                      # any number of EMA periods to overlay
ema_colors     = ["green", "orange", "blue", "purple"]  # one colour per period
initial_window = pd.Timedelta(days=3)                   # default zoom on load
chart_height   = 900

In [ ]:
# Candles + EMA overlays. The data is timestamp-indexed (engine contract), so the
# x-axis is df.index; EMAs use engine.indicators.ema for consistency with the analysis.
fig = go.Figure()
fig.add_trace(go.Candlestick(
    x=df.index,
    open=df["open"], high=df["high"], low=df["low"], close=df["close"],
    name=SYMBOL,
))
for period, color in zip(ema_periods, ema_colors):
    fig.add_trace(go.Scattergl(
        x=df.index,
        y=ema(df["close"], period),
        line=dict(color=color, width=1.5),
        name=f"EMA {period}",
    ))

initial_start = df.index[-1] - initial_window
initial_end   = df.index[-1]
ema_label = " & ".join(f"EMA {p}" for p in ema_periods)

fig.update_layout(
    title=f"{SYMBOL} — {ema_label}",
    height=chart_height, template="plotly_white",
    xaxis=dict(
        range=[initial_start, initial_end],
        rangeslider=dict(visible=True, thickness=0.04),
        rangeselector=dict(buttons=[
            dict(count=1, label="1d", step="day",   stepmode="backward"),
            dict(count=7, label="1w", step="day",   stepmode="backward"),
            dict(count=1, label="1m", step="month", stepmode="backward"),
            dict(count=3, label="3m", step="month", stepmode="backward"),
            dict(step="all", label="All"),
        ]),
    ),
    yaxis=dict(autorange=True, fixedrange=False),
)
fig.show()

## Conclusion

How to read the rankings:
- EMAs with high resistance ratios are candidates for the short side; EMAs with high support ratios are candidates for the long side.
- The strongest research candidates combine a high ratio with a meaningful sample size. An EMA with a slightly higher ratio but few touches may have been a secondary level in this window — interpret with caution.
- Overlay the chosen EMAs on the chart above to check they line up with swing highs and lows. If they do, those EMAs coincided with dynamic support/resistance during the historical window. This describes past data only and does not imply the levels will continue to hold.

Bridge to the strategy:

The cell below takes the top-ranked support and resistance EMAs (those passing both filters) and builds the EmaTouchParams you would run them with. Per-side entry EMAs (ema_touch_period_long for the support EMA, ema_touch_period_short for the resistance EMA) feed ema_touch directly; a None falls back to the symmetric ema_touch_period. The same delta and delta_mode used for the ranking are carried over.

Plug the printed params into the ema_touch notebook's STRATEGY_OVERRIDES, or set them as the defaults in strategy_configurator.py's EmaTouchParams.

In [ ]:
# Pick the top-ranked EMA (by hold rate, passing both filters) per direction and
# show the EmaTouchParams that would trade them.
def best_ema(test_col, held_col, min_touches):
    r = filter_by_cross_rate(result, MAX_CROSS_RATE)
    r = r[r[test_col] >= min_touches].copy()
    if r.empty:
        return None
    r["rate"] = r[held_col] / r[test_col].replace(0, np.nan)
    return int(r.sort_values("rate", ascending=False).iloc[0]["ema"])

best_support    = best_ema("support_test", "support_held", MIN_TOUCHES_SUP)
best_resistance = best_ema("resistance_test", "resistance_held", MIN_TOUCHES_RES)
print(f"Top support EMA (longs):     {best_support}")
print(f"Top resistance EMA (shorts): {best_resistance}")

suggested_params = dataclasses.replace(
    params_for("ema_touch"),
    ema_touch_period_long  = best_support,
    ema_touch_period_short = best_resistance,
    ema_touch_delta        = DELTA,
    ema_touch_delta_mode   = DELTA_MODE,
)
print()
print("Suggested EmaTouchParams:")
print(suggested_params)